# Cascada Thoracolumbar Mejorada y Explicada - Colab

Este notebook es una **nueva iteracion experimental** del entrenamiento multiclase
thoracolumbar. No reemplaza el notebook anterior: lo complementa como parte del
historial de exploracion del proyecto.

## Objetivo

Entrenar un modelo que identifique vertebras toracicas y lumbares usando una
cascada de dos etapas:

1. un modelo binario ya entrenado localiza la columna
2. a partir de esa localizacion se define una ROI espinal
3. una red multiclase aprende a separar `background + T1..T12 + L1..L5`

## Que cambia respecto al experimento anterior

En esta version reforzamos varios puntos que se detectaron como limitantes:

- usamos por defecto el subset `partial` para aumentar cantidad de muestras
- aumentamos la resolucion de la ROI multiclase
- mantenemos ROI basada en binario real para train y ROI predicha para val/test
- agregamos canales de coordenadas `y/x` para dar contexto anatomico
- ampliamos el padding de la ROI para reducir recortes accidentales
- usamos pesos de clase suavizados para estabilizar el entrenamiento
- agregamos scheduler y early stopping
- documentamos cada seccion para que este notebook sirva tambien como insumo
  del documento de decisiones del proyecto

## 0. Preparacion de Colab

Esta primera celda monta Google Drive y posiciona el notebook en la carpeta
raiz del proyecto.

Antes de ejecutarla:

1. sube la carpeta completa del proyecto a Google Drive
2. conserva la estructura de carpetas
3. ajusta `PROJECT_ROOT` si tu ruta en Drive es distinta

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/DataRadriografias")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"No existe la carpeta: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

## 1. Librerias, semillas y configuracion del experimento

Esta celda centraliza la configuracion del experimento.

Puntos importantes:

- `MULTICLASS_SUBSET = 'partial'` usa mas muestras que `core`
- `IMG_SIZE_MULTICLASS` sube la resolucion de la ROI
- `UNET_BASE` aumenta la capacidad del modelo
- `PATIENCE_EARLY_STOP` evita entrenamientos muy largos sin mejora real

In [ ]:
from __future__ import annotations

import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd()
DATASET_ROOT = ROOT / 'data' / 'Scoliosis_Dataset'

def resolve_existing(*relative_candidates: str) -> Path:
    search_roots = [ROOT, ROOT / 'reports']
    for base in search_roots:
        for rel in relative_candidates:
            candidate = base / rel
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'No se encontro ninguno de estos archivos: {relative_candidates}')


INDEX_PATH = resolve_existing('indice_dataset.csv')
DICT_PATH = resolve_existing('diccionario_etiquetas_T1_T12_L1_L5.json')
MANIFEST_PATH = resolve_existing('analysis_outputs/training_manifest_thoracolumbar.csv')
BINARY_GROUP_MAP_PATH = resolve_existing('analysis_outputs/training_runs_binary_thoracolumbar/binary_spine_group_partition_map.csv')
BINARY_MODEL_PATH = resolve_existing('models/binary_spine_thoracolumbar_best.pt')
OUTPUT_DIR = ROOT / 'analysis_outputs' / 'training_runs_cascade_thoracolumbar_explained'
MODEL_DIR = ROOT / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for path in [INDEX_PATH, DICT_PATH, MANIFEST_PATH, BINARY_GROUP_MAP_PATH, BINARY_MODEL_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'No existe archivo requerido: {path}')

def resolve_dataset_path(rel: str) -> str:
    rel_path = Path(str(rel).replace('\', '/'))
    if rel_path.is_absolute():
        return str(rel_path)
    candidates = [
        ROOT / rel_path,
        DATASET_ROOT / rel_path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate.resolve())
    return str((DATASET_ROOT / rel_path).resolve())

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'
USE_AMP = DEVICE.type == 'cuda'

IMG_SIZE_BINARY = (512, 256)
IMG_SIZE_MULTICLASS = (640, 320)
BATCH_SIZE = 4
NUM_WORKERS = 2
LR = 5e-4
WEIGHT_DECAY = 1e-4
MULTICLASS_EPOCHS = 45
PATIENCE_EARLY_STOP = 10
MULTICLASS_SUBSET = 'partial'   # opciones: 'core' o 'partial'
IGNORE_INDEX = 255
BINARY_THRESHOLD = 0.50
ROI_PAD_X = 28
ROI_PAD_Y = 44
ROI_JITTER_X = 12
ROI_JITTER_Y = 16
MIN_FOREGROUND_PIXELS = 24
HFLIP_PROB = 0.50
INTENSITY_JITTER_BRIGHTNESS = 0.12
INTENSITY_JITTER_CONTRAST = 0.12
UNET_BASE = 48
UNET_DROPOUT = 0.10

EXPERIMENT_CONFIG = {
    'multiclass_subset': MULTICLASS_SUBSET,
    'img_size_binary': IMG_SIZE_BINARY,
    'img_size_multiclass': IMG_SIZE_MULTICLASS,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'multiclass_epochs': MULTICLASS_EPOCHS,
    'patience_early_stop': PATIENCE_EARLY_STOP,
    'binary_threshold': BINARY_THRESHOLD,
    'roi_pad_x': ROI_PAD_X,
    'roi_pad_y': ROI_PAD_Y,
    'roi_jitter_x': ROI_JITTER_X,
    'roi_jitter_y': ROI_JITTER_Y,
    'min_foreground_pixels': MIN_FOREGROUND_PIXELS,
    'hflip_prob': HFLIP_PROB,
    'intensity_jitter_brightness': INTENSITY_JITTER_BRIGHTNESS,
    'intensity_jitter_contrast': INTENSITY_JITTER_CONTRAST,
    'unet_base': UNET_BASE,
    'unet_dropout': UNET_DROPOUT,
    'device': str(DEVICE),
}

print('ROOT:', ROOT)
print('DEVICE:', DEVICE)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('CONFIG:', json.dumps(EXPERIMENT_CONFIG, indent=2))

## 2. Carga de metadata y armado de la tabla de entrenamiento

En esta celda:

- leemos el indice oficial del dataset
- leemos el manifest generado en el notebook de analisis
- leemos el mapeo de particiones train/val/test producido por la etapa binaria
- filtramos el subset elegido (`core` o `partial`)

Esto garantiza consistencia entre las etapas del pipeline.

In [ ]:
index_df_raw = pd.read_csv(INDEX_PATH)
manifest_df = pd.read_csv(MANIFEST_PATH)
group_map_df = pd.read_csv(BINARY_GROUP_MAP_PATH)
with open(DICT_PATH, 'r', encoding='utf-8') as f:
    labels_dict = json.load(f)

index_col_map = {
    'grupo': 'split',
    'imagen': 'image',
    'id_paciente': 'patient_id',
    'ruta_radiografia': 'radiograph_path',
    'ruta_mascara_binaria': 'label_binary_path',
    'ruta_mascara_multiclase_id_png': 'multiclass_id_png',
}
index_df = index_df_raw.rename(columns=index_col_map).copy()

final_multiclass_map = {int(k): v for k, v in labels_dict['mascara_multiclase_id_png'].items()}
class_names = [final_multiclass_map[i] for i in range(len(final_multiclass_map))]
num_classes = len(class_names)
valid_multiclass_ids = set(range(num_classes))

join_cols = ['split', 'image', 'patient_id', 'radiograph_path']
dataset_subset = index_df[join_cols + ['label_binary_path', 'multiclass_id_png']].copy()
train_table = manifest_df.merge(dataset_subset, on=join_cols, how='left', suffixes=('', '_idx'))
train_table['multiclass_mask_path'] = train_table['mask_path'].fillna(train_table['multiclass_id_png'])
train_table['radiograph_path_abs'] = train_table['radiograph_path'].apply(resolve_dataset_path)
train_table['binary_mask_path_abs'] = train_table['label_binary_path'].apply(resolve_dataset_path)
train_table['multiclass_mask_path_abs'] = train_table['multiclass_mask_path'].apply(resolve_dataset_path)

for col in [
    'usable_for_thoracolumbar_core',
    'usable_for_thoracolumbar_partial',
    'needs_annotation_review',
]:
    if col in train_table.columns:
        train_table[col] = train_table[col].map(
            lambda x: x if isinstance(x, bool) else str(x).strip().lower() == 'true'
        )

group_partition_map = group_map_df.drop_duplicates().set_index('group_id_for_split')['partition'].to_dict()
train_table['partition'] = train_table['group_id_for_split'].map(group_partition_map)
if train_table['partition'].isna().any():
    missing_groups = sorted(train_table.loc[train_table['partition'].isna(), 'group_id_for_split'].dropna().unique().tolist())
    raise ValueError(f'Hay grupos sin split asignado desde el notebook binario: {missing_groups[:10]}')

multiclass_flag = 'usable_for_thoracolumbar_core' if MULTICLASS_SUBSET == 'core' else 'usable_for_thoracolumbar_partial'
multiclass_df = train_table.loc[train_table[multiclass_flag] & ~train_table['needs_annotation_review']].copy()

print('Total muestras del proyecto:', len(train_table))
print('Subset multiclase seleccionado:', MULTICLASS_SUBSET)
print('Muestras en subset multiclase:', len(multiclass_df))
display(multiclass_df.groupby(['partition', 'split']).size().rename('images').reset_index())
display(multiclass_df.head())

## 3. Auditoria rapida del subset seleccionado

Esta celda sirve como control de calidad antes de entrenar.

Revisamos:

- cuantas imagenes quedan por particion
- cuantos ejemplos visibles hay por vertebra y por particion
- si alguna vertebra queda demasiado subrepresentada en train/val/test

In [ ]:
import re

present_cols = [
    col
    for col in multiclass_df.columns
    if re.fullmatch(r'present_(T\d+|L\d+)', col) is not None
]

partition_presence_rows = []
for partition_name, partition_df in multiclass_df.groupby('partition'):
    row = {'partition': partition_name, 'images': len(partition_df)}
    for col in present_cols:
        row[col] = int(pd.to_numeric(partition_df[col], errors='coerce').fillna(0).sum())
    partition_presence_rows.append(row)

partition_presence_df = pd.DataFrame(partition_presence_rows)
long_partition_presence_df = partition_presence_df.melt(
    id_vars=['partition', 'images'],
    value_vars=present_cols,
    var_name='vertebra_col',
    value_name='images_with_label',
)
long_partition_presence_df['vertebra_label'] = long_partition_presence_df['vertebra_col'].str.replace('present_', '', regex=False)
long_partition_presence_df['coverage_ratio_in_partition'] = (
    long_partition_presence_df['images_with_label'] / long_partition_presence_df['images']
)

low_coverage_df = long_partition_presence_df.loc[
    long_partition_presence_df['coverage_ratio_in_partition'] < 0.20
].sort_values(['partition', 'coverage_ratio_in_partition', 'vertebra_label'])

display(partition_presence_df)
display(low_coverage_df.head(30))

## 4. Utilidades de imagen, ROI, augmentacion y canales posicionales

Esta es una de las celdas mas importantes del notebook.

Logica principal:

- la imagen entra en escala de grises
- la ROI se obtiene a partir del binario
- en `train` usamos ROI basada en mascara binaria real
- en `val/test` usamos ROI predicha por el modelo binario
- agregamos dos canales extra con coordenadas `y` y `x`

Esos canales posicionales le dan a la red una pista anatomica adicional,
lo cual es muy util porque T1 y L5 no se diferencian solo por textura.

In [ ]:
def read_gray(path: str) -> np.ndarray:
    return np.array(Image.open(path).convert('L'))


def resize_image(arr: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    return np.array(Image.fromarray(arr).resize((size[1], size[0]), resample=Image.BILINEAR))


def resize_mask(arr: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    return np.array(Image.fromarray(arr.astype(np.uint8)).resize((size[1], size[0]), resample=Image.NEAREST))


def build_binary_mask(path: str, size: tuple[int, int] | None = None) -> np.ndarray:
    mask = read_gray(path)
    mask = (mask >= 127).astype(np.uint8)
    if size is not None:
        mask = resize_mask(mask, size)
    return mask


def build_multiclass_mask(path: str, size: tuple[int, int] | None = None) -> np.ndarray:
    raw = np.array(Image.open(path), dtype=np.int32)
    out = np.zeros_like(raw, dtype=np.uint8)
    valid_mask = np.isin(raw, list(valid_multiclass_ids))
    out[~valid_mask] = IGNORE_INDEX
    out[valid_mask] = raw[valid_mask].astype(np.uint8)
    if size is not None:
        out = resize_mask(out, size)
    return out


def bbox_from_mask(mask: np.ndarray, min_foreground_pixels: int = 24) -> tuple[int, int, int, int] | None:
    ys, xs = np.where(mask > 0)
    if len(xs) < min_foreground_pixels:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def clamp_bbox(bbox: tuple[int, int, int, int], image_shape: tuple[int, int]) -> tuple[int, int, int, int]:
    h, w = image_shape
    x0, y0, x1, y1 = bbox
    x0 = max(0, min(x0, w - 1))
    y0 = max(0, min(y0, h - 1))
    x1 = max(x0 + 1, min(x1, w))
    y1 = max(y0 + 1, min(y1, h))
    return x0, y0, x1, y1


def expand_bbox(bbox: tuple[int, int, int, int], image_shape: tuple[int, int], pad_x: int = 18, pad_y: int = 30) -> tuple[int, int, int, int]:
    x0, y0, x1, y1 = bbox
    return clamp_bbox((x0 - pad_x, y0 - pad_y, x1 + pad_x, y1 + pad_y), image_shape)


def jitter_bbox(bbox: tuple[int, int, int, int], image_shape: tuple[int, int], max_jitter_x: int = 10, max_jitter_y: int = 14) -> tuple[int, int, int, int]:
    x0, y0, x1, y1 = bbox
    dx0 = np.random.randint(-max_jitter_x, max_jitter_x + 1)
    dx1 = np.random.randint(-max_jitter_x, max_jitter_x + 1)
    dy0 = np.random.randint(-max_jitter_y, max_jitter_y + 1)
    dy1 = np.random.randint(-max_jitter_y, max_jitter_y + 1)
    return clamp_bbox((x0 + dx0, y0 + dy0, x1 + dx1, y1 + dy1), image_shape)


def crop_array(arr: np.ndarray, bbox: tuple[int, int, int, int]) -> np.ndarray:
    x0, y0, x1, y1 = bbox
    return arr[y0:y1, x0:x1]


def scale_bbox(bbox: tuple[int, int, int, int], src_shape: tuple[int, int], dst_shape: tuple[int, int]) -> tuple[int, int, int, int]:
    src_h, src_w = src_shape
    dst_h, dst_w = dst_shape
    x0, y0, x1, y1 = bbox
    sx = dst_w / src_w
    sy = dst_h / src_h
    scaled = (
        int(round(x0 * sx)),
        int(round(y0 * sy)),
        int(round(x1 * sx)),
        int(round(y1 * sy)),
    )
    return clamp_bbox(scaled, dst_shape)


def full_image_bbox(image_shape: tuple[int, int]) -> tuple[int, int, int, int]:
    h, w = image_shape
    return 0, 0, w, h


def build_roi_record_from_binary_mask(binary_mask: np.ndarray, image_shape: tuple[int, int], roi_source: str) -> dict:
    bbox = bbox_from_mask(binary_mask, min_foreground_pixels=MIN_FOREGROUND_PIXELS)
    if bbox is None:
        bbox = full_image_bbox(image_shape)
        roi_source = f'{roi_source}_fallback_full_image'
    else:
        bbox = expand_bbox(bbox, image_shape=image_shape, pad_x=ROI_PAD_X, pad_y=ROI_PAD_Y)
    x0, y0, x1, y1 = bbox
    return {
        'bbox_x0': x0,
        'bbox_y0': y0,
        'bbox_x1': x1,
        'bbox_y1': y1,
        'bbox_width': x1 - x0,
        'bbox_height': y1 - y0,
        'roi_source': roi_source,
    }


def apply_intensity_jitter(image_2d: np.ndarray, brightness: float = 0.12, contrast: float = 0.12) -> np.ndarray:
    gain = 1.0 + np.random.uniform(-contrast, contrast)
    bias = np.random.uniform(-brightness, brightness)
    out = image_2d * gain + bias
    return np.clip(out, 0.0, 1.0)


def normalize_image(image_2d: np.ndarray) -> np.ndarray:
    mean = float(image_2d.mean())
    std = float(image_2d.std())
    if std < 1e-6:
        return image_2d - mean
    return (image_2d - mean) / std


def build_coordinate_channels(height: int, width: int) -> np.ndarray:
    y_coords = np.linspace(0.0, 1.0, height, dtype=np.float32)[:, None]
    x_coords = np.linspace(0.0, 1.0, width, dtype=np.float32)[None, :]
    y_map = np.repeat(y_coords, width, axis=1)
    x_map = np.repeat(x_coords, height, axis=0)
    return np.stack([y_map, x_map], axis=0)


def maybe_horizontal_flip(image_2d: np.ndarray, mask_2d: np.ndarray, probability: float = 0.50) -> tuple[np.ndarray, np.ndarray]:
    if np.random.rand() < probability:
        image_2d = np.ascontiguousarray(np.fliplr(image_2d))
        mask_2d = np.ascontiguousarray(np.fliplr(mask_2d))
    return image_2d, mask_2d


def prepare_multiclass_cascade_sample(
    row: pd.Series,
    output_size: tuple[int, int],
    roi_mode: str,
    roi_lookup: dict | None = None,
    apply_jitter: bool = False,
    apply_intensity_aug: bool = False,
    apply_hflip: bool = False,
) -> dict:
    image_raw = read_gray(row['radiograph_path_abs'])
    binary_raw = build_binary_mask(row['binary_mask_path_abs'], size=None)
    multiclass_raw = build_multiclass_mask(row['multiclass_mask_path_abs'], size=None)
    image_shape = image_raw.shape

    if roi_mode == 'gt_binary':
        roi_meta = build_roi_record_from_binary_mask(binary_raw, image_shape=image_shape, roi_source='gt_binary')
    elif roi_mode == 'pred_binary':
        if roi_lookup is None:
            raise ValueError('roi_lookup es obligatorio para roi_mode=pred_binary')
        roi_meta = roi_lookup.get(row['unique_sample_id'])
        if roi_meta is None:
            roi_meta = build_roi_record_from_binary_mask(
                np.zeros_like(binary_raw),
                image_shape=image_shape,
                roi_source='pred_binary_missing',
            )
    else:
        raise ValueError(f'roi_mode no soportado: {roi_mode}')

    bbox = (
        int(roi_meta['bbox_x0']),
        int(roi_meta['bbox_y0']),
        int(roi_meta['bbox_x1']),
        int(roi_meta['bbox_y1']),
    )
    if apply_jitter:
        bbox = jitter_bbox(bbox, image_shape=image_shape, max_jitter_x=ROI_JITTER_X, max_jitter_y=ROI_JITTER_Y)

    image_crop = crop_array(image_raw, bbox)
    multiclass_crop = crop_array(multiclass_raw, bbox)
    image_crop = resize_image(image_crop, output_size).astype(np.float32) / 255.0
    multiclass_crop = resize_mask(multiclass_crop, output_size).astype(np.int64)

    if apply_hflip:
        image_crop, multiclass_crop = maybe_horizontal_flip(image_crop, multiclass_crop, probability=HFLIP_PROB)
    if apply_intensity_aug:
        image_crop = apply_intensity_jitter(
            image_crop,
            brightness=INTENSITY_JITTER_BRIGHTNESS,
            contrast=INTENSITY_JITTER_CONTRAST,
        )

    image_crop = normalize_image(image_crop)
    coord_channels = build_coordinate_channels(output_size[0], output_size[1])
    image_channels = np.concatenate([np.expand_dims(image_crop, axis=0), coord_channels], axis=0)

    return {
        'image': image_channels,
        'mask': multiclass_crop,
        'bbox': bbox,
        'roi_source': roi_meta['roi_source'],
    }


class CascadedThoracolumbarDataset(Dataset):
    def __init__(
        self,
        table: pd.DataFrame,
        image_size: tuple[int, int],
        roi_mode: str,
        roi_lookup: dict | None = None,
        apply_jitter: bool = False,
        apply_intensity_aug: bool = False,
        apply_hflip: bool = False,
    ):
        self.table = table.reset_index(drop=True).copy()
        self.image_size = image_size
        self.roi_mode = roi_mode
        self.roi_lookup = roi_lookup
        self.apply_jitter = apply_jitter
        self.apply_intensity_aug = apply_intensity_aug
        self.apply_hflip = apply_hflip

    def __len__(self) -> int:
        return len(self.table)

    def __getitem__(self, idx: int) -> dict:
        row = self.table.iloc[idx]
        sample = prepare_multiclass_cascade_sample(
            row=row,
            output_size=self.image_size,
            roi_mode=self.roi_mode,
            roi_lookup=self.roi_lookup,
            apply_jitter=self.apply_jitter,
            apply_intensity_aug=self.apply_intensity_aug,
            apply_hflip=self.apply_hflip,
        )
        return {
            'image': torch.tensor(sample['image'], dtype=torch.float32),
            'mask': torch.tensor(sample['mask'], dtype=torch.int64),
            'sample_id': row['unique_sample_id'],
            'roi_source': sample['roi_source'],
        }

## 5. Modelo, perdidas y metricas

En esta version usamos una U-Net algo mas potente que la pequena base:

- `in_channels = 3` porque entra intensidad + coordenada y + coordenada x
- `base = 48` para ganar capacidad representacional
- `dropout` en el bottleneck para regularizacion suave

Las metricas clave para seguir son:

- `macro_dice_fg`: promedio Dice sobre vertebras, sin contar fondo
- `macro_iou_fg`: promedio IoU sobre vertebras
- `pixel_accuracy`: utilidad secundaria, no debe tomarse sola

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, c_in: int, c_out: int, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class UNetEnhanced(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, base: int = 48, dropout: float = 0.10):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.e1 = DoubleConv(in_channels, base, dropout=0.0)
        self.e2 = DoubleConv(base, base * 2, dropout=0.0)
        self.e3 = DoubleConv(base * 2, base * 4, dropout=0.0)
        self.e4 = DoubleConv(base * 4, base * 8, dropout=dropout * 0.5)
        self.b = DoubleConv(base * 8, base * 16, dropout=dropout)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, kernel_size=2, stride=2)
        self.d4 = DoubleConv(base * 16, base * 8, dropout=dropout * 0.5)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, kernel_size=2, stride=2)
        self.d3 = DoubleConv(base * 8, base * 4, dropout=0.0)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.d2 = DoubleConv(base * 4, base * 2, dropout=0.0)
        self.u1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.d1 = DoubleConv(base * 2, base, dropout=0.0)
        self.head = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], dim=1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.head(d1)


def dice_loss_multiclass(logits: torch.Tensor, targets: torch.Tensor, num_classes: int, ignore_index: int = 255, eps: float = 1e-6) -> torch.Tensor:
    valid = targets != ignore_index
    targets_safe = targets.clone()
    targets_safe[~valid] = 0
    probs = torch.softmax(logits, dim=1)
    target_1h = F.one_hot(targets_safe, num_classes=num_classes).permute(0, 3, 1, 2).float()
    valid_mask = valid.unsqueeze(1)
    probs = probs * valid_mask
    target_1h = target_1h * valid_mask
    intersection = (probs * target_1h).sum(dim=(0, 2, 3))
    denominator = probs.sum(dim=(0, 2, 3)) + target_1h.sum(dim=(0, 2, 3))
    dice = (2.0 * intersection + eps) / (denominator + eps)
    return 1.0 - dice[1:].mean()


def evaluate_multiclass(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[dict[str, float], pd.DataFrame]:
    model.eval()
    total_loss = 0.0
    total_batches = 0
    correct = 0.0
    total_valid_pixels = 0.0
    intersection = np.zeros(num_classes, dtype=np.float64)
    pred_area = np.zeros(num_classes, dtype=np.float64)
    target_area = np.zeros(num_classes, dtype=np.float64)

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(DEVICE, non_blocking=True)
            targets = batch['mask'].to(DEVICE, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                logits = model(images)
                loss = loss_fn(logits, targets) + dice_loss_multiclass(
                    logits,
                    targets,
                    num_classes=num_classes,
                    ignore_index=IGNORE_INDEX,
                )

            total_loss += float(loss.item())
            total_batches += 1
            preds = torch.argmax(logits, dim=1)
            valid = targets != IGNORE_INDEX
            correct += float(((preds == targets) & valid).sum().item())
            total_valid_pixels += float(valid.sum().item())
            preds_np = preds[valid].detach().cpu().numpy()
            targets_np = targets[valid].detach().cpu().numpy()

            for class_idx in range(num_classes):
                pred_mask = preds_np == class_idx
                target_mask = targets_np == class_idx
                intersection[class_idx] += np.logical_and(pred_mask, target_mask).sum()
                pred_area[class_idx] += pred_mask.sum()
                target_area[class_idx] += target_mask.sum()

    dice = (2.0 * intersection + 1e-6) / (pred_area + target_area + 1e-6)
    iou = (intersection + 1e-6) / (pred_area + target_area - intersection + 1e-6)
    per_class_df = pd.DataFrame({
        'class_id': np.arange(num_classes),
        'class_name': class_names,
        'pred_pixels': pred_area,
        'target_pixels': target_area,
        'dice': dice,
        'iou': iou,
    })
    per_class_df['region'] = per_class_df['class_name'].map(
        lambda x: 'background' if x == 'background' else ('thoracic' if x.startswith('T') else 'lumbar')
    )
    fg_df = per_class_df.loc[per_class_df['class_id'] > 0].copy()

    metrics = {
        'loss': total_loss / max(total_batches, 1),
        'pixel_accuracy': (correct + 1e-6) / (total_valid_pixels + 1e-6),
        'macro_dice_fg': float(fg_df['dice'].mean()),
        'macro_iou_fg': float(fg_df['iou'].mean()),
        'macro_dice_thoracic': float(fg_df.loc[fg_df['region'] == 'thoracic', 'dice'].mean()),
        'macro_dice_lumbar': float(fg_df.loc[fg_df['region'] == 'lumbar', 'dice'].mean()),
    }
    return metrics, per_class_df


def plot_history(history_df: pd.DataFrame, title: str) -> None:
    metric_cols = [col for col in history_df.columns if col != 'epoch']
    ncols = 2
    nrows = int(math.ceil(len(metric_cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, max(4, 4 * nrows)))
    axes = np.atleast_1d(axes).ravel()
    for ax, col in zip(axes, metric_cols):
        ax.plot(history_df['epoch'], history_df[col], marker='o')
        ax.set_title(col)
        ax.set_xlabel('epoch')
        ax.grid(True, alpha=0.3)
    for ax in axes[len(metric_cols):]:
        ax.axis('off')
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

## 6. Prediccion de ROI usando el modelo binario y diagnostico de recortes

Antes de entrenar la red multiclase vale la pena auditar la ROI:

- cuantas veces el binario falla y se usa la imagen completa
- que tan grandes son las cajas
- si la nueva configuracion de padding esta siendo mas conservadora

In [ ]:
class DoubleConvBinary(nn.Module):
    def __init__(self, c_in: int, c_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class BinaryUNetSmall(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, base: int = 32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.e1 = DoubleConvBinary(in_channels, base)
        self.e2 = DoubleConvBinary(base, base * 2)
        self.e3 = DoubleConvBinary(base * 2, base * 4)
        self.e4 = DoubleConvBinary(base * 4, base * 8)
        self.b = DoubleConvBinary(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, kernel_size=2, stride=2)
        self.d4 = DoubleConvBinary(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, kernel_size=2, stride=2)
        self.d3 = DoubleConvBinary(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.d2 = DoubleConvBinary(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.d1 = DoubleConvBinary(base * 2, base)
        self.head = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], dim=1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.head(d1)


def predict_binary_rois_for_table(model: nn.Module, table: pd.DataFrame, input_size: tuple[int, int], threshold: float = 0.50) -> pd.DataFrame:
    records = []
    model.eval()
    with torch.no_grad():
        for _, row in table.reset_index(drop=True).iterrows():
            image_raw = read_gray(row['radiograph_path_abs'])
            original_shape = image_raw.shape
            image_resized = resize_image(image_raw, input_size).astype(np.float32) / 255.0
            image_resized = torch.tensor(image_resized[None, None, ...], dtype=torch.float32, device=DEVICE)
            logits = model(image_resized)
            pred_mask_small = (torch.sigmoid(logits)[0, 0].detach().cpu().numpy() >= threshold).astype(np.uint8)
            bbox_small = bbox_from_mask(pred_mask_small, min_foreground_pixels=MIN_FOREGROUND_PIXELS)

            if bbox_small is None:
                roi_meta = build_roi_record_from_binary_mask(
                    np.zeros(original_shape, dtype=np.uint8),
                    image_shape=original_shape,
                    roi_source='pred_binary_empty',
                )
            else:
                bbox_original = scale_bbox(bbox_small, src_shape=input_size, dst_shape=original_shape)
                bbox_original = expand_bbox(
                    bbox_original,
                    image_shape=original_shape,
                    pad_x=ROI_PAD_X,
                    pad_y=ROI_PAD_Y,
                )
                x0, y0, x1, y1 = bbox_original
                roi_meta = {
                    'bbox_x0': x0,
                    'bbox_y0': y0,
                    'bbox_x1': x1,
                    'bbox_y1': y1,
                    'bbox_width': x1 - x0,
                    'bbox_height': y1 - y0,
                    'roi_source': 'pred_binary',
                }

            records.append({'unique_sample_id': row['unique_sample_id'], **roi_meta})
    return pd.DataFrame(records)


binary_model = BinaryUNetSmall(in_channels=1, out_channels=1).to(DEVICE)
binary_model.load_state_dict(torch.load(BINARY_MODEL_PATH, map_location=DEVICE))
binary_model.eval()

roi_df = predict_binary_rois_for_table(
    binary_model,
    multiclass_df,
    input_size=IMG_SIZE_BINARY,
    threshold=BINARY_THRESHOLD,
)
roi_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_binary_rois.csv'
roi_df.to_csv(roi_path, index=False)
roi_lookup = roi_df.set_index('unique_sample_id').to_dict(orient='index')

roi_summary_df = pd.DataFrame(
    [
        {'metric': 'total_samples', 'value': len(roi_df)},
        {'metric': 'pred_binary_empty_or_fallback', 'value': int(roi_df['roi_source'].astype(str).str.contains('fallback').sum())},
        {'metric': 'mean_bbox_width', 'value': float(roi_df['bbox_width'].mean())},
        {'metric': 'mean_bbox_height', 'value': float(roi_df['bbox_height'].mean())},
        {'metric': 'median_bbox_width', 'value': float(roi_df['bbox_width'].median())},
        {'metric': 'median_bbox_height', 'value': float(roi_df['bbox_height'].median())},
    ]
)

display(roi_summary_df)
display(roi_df.head())

## 7. Dataloaders y pesos de clase suavizados

En la version anterior los pesos por clase funcionaron, pero aun habia
inestabilidad. Aqui usamos una variante mas suave:

- en vez de usar inversa completa de frecuencia
- usamos raiz cuadrada de la inversa

Eso sigue ayudando a las clases menos frecuentes, pero reduce el riesgo
de que unas pocas clases dominen demasiado la funcion de costo.

In [ ]:
def estimate_multiclass_class_weights(table: pd.DataFrame) -> tuple[torch.Tensor, pd.DataFrame]:
    counts = np.ones(num_classes, dtype=np.float64)
    for _, row in table.reset_index(drop=True).iterrows():
        sample = prepare_multiclass_cascade_sample(
            row=row,
            output_size=IMG_SIZE_MULTICLASS,
            roi_mode='gt_binary',
            roi_lookup=None,
            apply_jitter=False,
            apply_intensity_aug=False,
            apply_hflip=False,
        )
        mask = sample['mask']
        valid = mask != IGNORE_INDEX
        bincount = np.bincount(mask[valid].ravel(), minlength=num_classes)
        counts += bincount

    weights = np.power(counts.sum() / counts, 0.5)
    weights = weights / weights.mean()
    weights = np.clip(weights, 0.50, 4.00)

    weights_df = pd.DataFrame({
        'class_id': np.arange(num_classes),
        'class_name': class_names,
        'pixel_count': counts,
        'weight': weights,
    })
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE), weights_df


train_ds = CascadedThoracolumbarDataset(
    multiclass_df.query("partition == 'train'"),
    image_size=IMG_SIZE_MULTICLASS,
    roi_mode='gt_binary',
    roi_lookup=None,
    apply_jitter=True,
    apply_intensity_aug=True,
    apply_hflip=True,
)
val_ds = CascadedThoracolumbarDataset(
    multiclass_df.query("partition == 'val'"),
    image_size=IMG_SIZE_MULTICLASS,
    roi_mode='pred_binary',
    roi_lookup=roi_lookup,
    apply_jitter=False,
    apply_intensity_aug=False,
    apply_hflip=False,
)
test_ds = CascadedThoracolumbarDataset(
    multiclass_df.query("partition == 'test'"),
    image_size=IMG_SIZE_MULTICLASS,
    roi_mode='pred_binary',
    roi_lookup=roi_lookup,
    apply_jitter=False,
    apply_intensity_aug=False,
    apply_hflip=False,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

class_weights, class_weights_df = estimate_multiclass_class_weights(multiclass_df.query("partition == 'train'"))
display(class_weights_df)

## 8. Entrenamiento con scheduler, AMP y early stopping

Esta es la celda principal del entrenamiento.

Mejoras incluidas:

- `torch.autocast` y `GradScaler` para aprovechar mejor GPU en Colab
- `ReduceLROnPlateau` para bajar learning rate si la validacion se estanca
- `early stopping` para cortar si ya no hay mejora
- guardado del mejor checkpoint segun `val_macro_dice_fg`

In [ ]:
multiclass_model = UNetEnhanced(
    in_channels=3,
    out_channels=num_classes,
    base=UNET_BASE,
    dropout=UNET_DROPOUT,
).to(DEVICE)
multiclass_optimizer = torch.optim.AdamW(multiclass_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
multiclass_loss_fn = nn.CrossEntropyLoss(weight=class_weights, ignore_index=IGNORE_INDEX)
multiclass_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    multiclass_optimizer,
    mode='max',
    factor=0.5,
    patience=4,
)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

history = []
best_state = None
best_val_macro_dice = -1.0
epochs_without_improvement = 0
start_time = time.time()

for epoch in range(1, MULTICLASS_EPOCHS + 1):
    multiclass_model.train()
    for batch in train_loader:
        images = batch['image'].to(DEVICE, non_blocking=True)
        targets = batch['mask'].to(DEVICE, non_blocking=True)
        multiclass_optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            logits = multiclass_model(images)
            loss = multiclass_loss_fn(logits, targets) + dice_loss_multiclass(
                logits,
                targets,
                num_classes=num_classes,
                ignore_index=IGNORE_INDEX,
            )

        scaler.scale(loss).backward()
        scaler.step(multiclass_optimizer)
        scaler.update()

    train_metrics, _ = evaluate_multiclass(multiclass_model, train_loader, loss_fn=multiclass_loss_fn)
    val_metrics, _ = evaluate_multiclass(multiclass_model, val_loader, loss_fn=multiclass_loss_fn)
    current_lr = float(multiclass_optimizer.param_groups[0]['lr'])

    history.append({
        'epoch': epoch,
        'lr': current_lr,
        'train_loss': train_metrics['loss'],
        'train_pixel_accuracy': train_metrics['pixel_accuracy'],
        'train_macro_dice_fg': train_metrics['macro_dice_fg'],
        'train_macro_iou_fg': train_metrics['macro_iou_fg'],
        'train_macro_dice_thoracic': train_metrics['macro_dice_thoracic'],
        'train_macro_dice_lumbar': train_metrics['macro_dice_lumbar'],
        'val_loss': val_metrics['loss'],
        'val_pixel_accuracy': val_metrics['pixel_accuracy'],
        'val_macro_dice_fg': val_metrics['macro_dice_fg'],
        'val_macro_iou_fg': val_metrics['macro_iou_fg'],
        'val_macro_dice_thoracic': val_metrics['macro_dice_thoracic'],
        'val_macro_dice_lumbar': val_metrics['macro_dice_lumbar'],
    })

    if val_metrics['macro_dice_fg'] > best_val_macro_dice:
        best_val_macro_dice = val_metrics['macro_dice_fg']
        best_state = {k: v.detach().cpu().clone() for k, v in multiclass_model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    multiclass_scheduler.step(val_metrics['macro_dice_fg'])

    print(
        f"[Cascade-Explained][Epoch {epoch:02d}/{MULTICLASS_EPOCHS}] "
        f"lr={current_lr:.6f} "
        f"train_macro_dice={train_metrics['macro_dice_fg']:.4f} "
        f"val_macro_dice={val_metrics['macro_dice_fg']:.4f}"
    )

    if epochs_without_improvement >= PATIENCE_EARLY_STOP:
        print(f"Early stopping activado en epoch {epoch}.")
        break

history_df = pd.DataFrame(history)
multiclass_model.load_state_dict(best_state)
test_metrics, per_class_df = evaluate_multiclass(multiclass_model, test_loader, loss_fn=multiclass_loss_fn)

model_path = MODEL_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_cascade_explained_best.pt'
history_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_history.csv'
metrics_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_test_metrics.csv'
per_class_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_per_class_metrics.csv'
class_weights_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_class_weights.csv'
experiment_config_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_experiment_config.json'

torch.save(multiclass_model.state_dict(), model_path)
history_df.to_csv(history_path, index=False)
pd.DataFrame([test_metrics]).to_csv(metrics_path, index=False)
per_class_df.to_csv(per_class_path, index=False)
class_weights_df.to_csv(class_weights_path, index=False)
experiment_config_path.write_text(json.dumps(EXPERIMENT_CONFIG, indent=2), encoding='utf-8')

print('Checkpoint guardado en:', model_path)
print('Tiempo total (min):', round((time.time() - start_time) / 60.0, 2))
print('Mejor val_macro_dice_fg:', round(best_val_macro_dice, 4))
display(history_df.tail())
display(pd.DataFrame([test_metrics]))
plot_history(history_df, f'Historia cascada thoracolumbar explicada - {MULTICLASS_SUBSET}')

## 9. Analisis de metricas por clase y por region anatomica

Esta celda nos ayuda a responder preguntas mas finas:

- cuales vertebras salen mejor
- si toracicas y lumbares se comportan parecido
- donde siguen estando los puntos debiles del modelo

In [ ]:
region_summary_df = (
    per_class_df.loc[per_class_df['class_id'] > 0]
    .groupby('region')[['dice', 'iou', 'target_pixels', 'pred_pixels']]
    .mean()
    .reset_index()
)
hardest_classes_df = per_class_df.loc[per_class_df['class_id'] > 0].sort_values('dice', ascending=True).copy()
easiest_classes_df = per_class_df.loc[per_class_df['class_id'] > 0].sort_values('dice', ascending=False).copy()

display(region_summary_df)
display(hardest_classes_df.head(8))
display(easiest_classes_df.head(8))

plt.figure(figsize=(12, 5))
plt.bar(per_class_df['class_name'], per_class_df['dice'])
plt.xticks(rotation=45)
plt.ylabel('Dice')
plt.title('Dice por clase')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 10. Revision cualitativa de predicciones

Ninguna metrica sustituye la inspeccion visual.

Aqui miramos algunos casos de test para responder:

- si la ROI realmente esta enfocando la columna
- si el modelo separa vertebras vecinas
- si los errores son de desplazamiento, fusion o ausencia

In [ ]:
def show_cascade_multiclass_predictions(model: nn.Module, dataset: Dataset, n: int = 4) -> None:
    model.eval()
    n = min(n, len(dataset))
    fig, axes = plt.subplots(n, 3, figsize=(13, 4 * n))
    axes = np.atleast_2d(axes)
    with torch.no_grad():
        for idx in range(n):
            sample = dataset[idx]
            image = sample['image'].unsqueeze(0).to(DEVICE)
            pred = torch.argmax(model(image), dim=1)[0].detach().cpu().numpy()
            target = sample['mask'].numpy().copy()
            target[target == IGNORE_INDEX] = 0
            axes[idx, 0].imshow(sample['image'][0].numpy(), cmap='gray')
            axes[idx, 0].set_title(f"ROI: {sample['sample_id']} | {sample['roi_source']}")
            axes[idx, 1].imshow(target, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
            axes[idx, 1].set_title('GT multiclase')
            axes[idx, 2].imshow(pred, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
            axes[idx, 2].set_title('Pred multiclase')
            for j in range(3):
                axes[idx, j].axis('off')
    plt.tight_layout()
    plt.show()


show_cascade_multiclass_predictions(multiclass_model, test_ds, n=4)

## 11. Resumen final del experimento

Esta celda deja una tabla breve con los hallazgos principales del run.
La idea es que luego pueda servir como material para el documento de
decisiones y resumen del proyecto.

In [ ]:
experiment_summary_df = pd.DataFrame(
    [
        {'metric': 'subset', 'value': MULTICLASS_SUBSET},
        {'metric': 'train_images', 'value': int((multiclass_df['partition'] == 'train').sum())},
        {'metric': 'val_images', 'value': int((multiclass_df['partition'] == 'val').sum())},
        {'metric': 'test_images', 'value': int((multiclass_df['partition'] == 'test').sum())},
        {'metric': 'best_val_macro_dice_fg', 'value': float(best_val_macro_dice)},
        {'metric': 'test_macro_dice_fg', 'value': float(test_metrics['macro_dice_fg'])},
        {'metric': 'test_macro_iou_fg', 'value': float(test_metrics['macro_iou_fg'])},
        {'metric': 'test_macro_dice_thoracic', 'value': float(test_metrics['macro_dice_thoracic'])},
        {'metric': 'test_macro_dice_lumbar', 'value': float(test_metrics['macro_dice_lumbar'])},
        {'metric': 'roi_fallback_count', 'value': int(roi_df['roi_source'].astype(str).str.contains('fallback').sum())},
    ]
)

summary_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_experiment_summary.csv'
experiment_summary_df.to_csv(summary_path, index=False)

display(experiment_summary_df)
print('Resumen guardado en:', summary_path)